Trying to create a personal 3D stitiching algorithm

In [17]:
import tifffile
from skimage.measure import regionprops, regionprops_table
import napari
import numpy as np

image = tifffile.imread(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\cropped\Frame-10.tif"
)
crc_image = image[:,0]
wt_image = image[:,1]
mask = tifffile.imread(
    r"C:\Users\6331823\Local SSD\Data_Elise\EM_Exp007_Imaging_20x_2025-11-21_Maria_Elise_14.47.50_CZC34578YN_F24\segmented\Frame-10_masks.tif"
)
print(mask.shape)
# to the mask, add pixel on the left side with value 0
mask = np.pad(mask, ((0, 0), (0, 0), (1, 0)), mode='constant', constant_values=0)
print(mask.shape)
unique_mask = []

number = 1
for z in range(mask.shape[0]):
    # go over every unique number in the mask and create a new slice with a new unique number 1 higher
    slice = mask[z]
    unique_slice = mask[z].copy()
    for val in range(1, slice.max() + 1):
        unique_slice[slice == val] = number
        number += 1
    unique_mask.append(unique_slice)
unique_mask = np.stack(unique_mask, axis=0)

(92, 528, 606)
(92, 528, 607)


In [12]:
viewer = napari.Viewer()
viewer.add_image(crc_image, name="crc")
viewer.add_labels(unique_mask, name="mask")

<Labels layer 'mask' at 0x1e4109b6ad0>

In [22]:
import pandas as pd

def properties_mask(image):

    # Get the properties of every mask in the image
    props = regionprops(image)

    data = []

    # For every mask found, get the label, centeroid, boundingbox, and volume
    for prop in props:
        label = prop.label
        centroid = prop.centroid  # (z, y, x)
        bounding_box = prop.bbox  # (min_z, min_y, min_x, max_z, max_y, max_x)
        volume = prop.area

        data.append(
            {
                "label": label,
                "z": centroid[0],
                "y": centroid[1],
                "x": centroid[2],
                "bounding_box": bounding_box,
                "volume": volume,
                "coords_2d": prop.coords[:, 1:],
            }
        )

    # Make a pandas data frame from the data
    df_props = pd.DataFrame(data)

    return df_props


def properties_channel(mask, image, type=None):

    # Get the properties of the mask for the channel you specified (image)
    props_channel = regionprops_table(
        mask, intensity_image=image, properties=["label", "mean_intensity"]
    )

    # Make this into a pandas dataframe and rename some variables to the name of the channel for later merging
    props_channel = pd.DataFrame(props_channel).rename(
        columns={
            "label": "label",
            "mean_intensity": f"raw_{type}",
        }
    )

    return props_channel

print(image.shape, wt_image.shape, crc_image.shape, unique_mask.shape)

props_mask = properties_mask(unique_mask)
props_wt = properties_channel(unique_mask, wt_image, type="wt")
props_crc = properties_channel(unique_mask, crc_image, type="crc")

# Merge all the dataframes together on the label
df_final = props_mask.merge(props_wt, on="label").merge(props_crc, on="label")

(92, 3, 528, 607) (92, 528, 607) (92, 528, 607) (92, 528, 607)


In [23]:
df_final

,label,z,y,x,bounding_box,volume,coords_2d,raw_wt,raw_crc
0,1,0.0,176.362637,248.417582,"(0, 172, 243, 1, 182, 255)",91.0,"[[172, 246], [172, 247], [172, 248], [172, 249...",114.802198,106.945055
1,2,0.0,202.131868,329.483516,"(0, 198, 324, 1, 208, 336)",91.0,"[[198, 327], [198, 328], [198, 329], [198, 330...",118.186813,107.351648
2,3,1.0,172.890244,247.634146,"(1, 166, 241, 2, 181, 255)",164.0,"[[166, 244], [166, 245], [166, 246], [166, 247...",118.231707,107.274390
3,4,1.0,202.820690,328.393103,"(1, 197, 321, 2, 209, 337)",145.0,"[[197, 327], [197, 328], [198, 324], [198, 325...",121.641379,106.420690
4,5,1.0,143.888889,276.211111,"(1, 139, 271, 2, 150, 282)",90.0,"[[139, 275], [139, 276], [139, 277], [140, 273...",115.055556,107.011111
...,...,...,...,...,...,...,...,...,...
7663,53730,91.0,266.166667,307.598765,"(91, 260, 299, 92, 273, 318)",162.0,"[[260, 311], [261, 308], [261, 309], [261, 310...",120.808642,106.395062
7664,53732,91.0,276.522388,285.776119,"(91, 271, 279, 92, 284, 293)",134.0,"[[271, 280], [271, 281], [271, 282], [271, 283...",120.664179,106.582090
7665,53733,91.0,297.800000,303.706667,"(91, 293, 299, 92, 303, 309)",75.0,"[[293, 304], [294, 301], [294, 302], [294, 303...",126.240000,105.853333
7666,53737,91.0,242.304348,425.468443,"(91, 226, 414, 92, 261, 440)",713.0,"[[226, 423], [226, 424], [226, 425], [226, 426...",115.890603,107.548387


In [53]:
def calculate_iou_2d(mask1_coords, mask2_coords):
    """Calculate IOU between two masks using their coordinates"""
    # Convert coordinates to sets for intersection/union
    set1 = set(map(tuple, mask1_coords))
    set2 = set(map(tuple, mask2_coords))

    intersection = len(set1 & set2)
    union = len(set1 | set2)

    return intersection / union if union > 0 else 0

In [68]:
# add flag column to df_final
df_final["3d_label"] = -1  # Use -1 to indicate unassigned
mask_3D = np.zeros_like(unique_mask)
unique_label = 0

# Pre-compute: group indices by z-slice for fast lookup
z_groups = {}
for idx, z_val in df_final["z"].items():
    z_int = int(round(z_val))
    if z_int not in z_groups:
        z_groups[z_int] = []
    z_groups[z_int].append(idx)

# Pre-compute: convert all coords to sets once (much faster)
coords_sets = {idx: set(map(tuple, coords)) for idx, coords in df_final["coords_2d"].items()}

def calculate_iou_fast(idx1, idx2):
    """Fast IOU using pre-computed sets"""
    set1 = coords_sets[idx1]
    set2 = coords_sets[idx2]
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union > 0 else 0

def stitch_forward(idx, current_label):
    """Recursively stitch a cell forward through z-slices"""
    row = df_final.loc[idx]
    current_z = int(round(row["z"]))
    
    # Assign the current label to this row
    df_final.at[idx, "3d_label"] = current_label
    
    # Fast lookup: get only indices in the next slice
    next_z = current_z + 1
    if next_z not in z_groups:
        return  # No next slice
    
    candidates = z_groups[next_z]
    
    # Vectorized distance calculation for all candidates
    x_dists = df_final.loc[candidates, "x"].values - row["x"]
    y_dists = df_final.loc[candidates, "y"].values - row["y"]
    distances = np.sqrt(x_dists**2 + y_dists**2)
    
    # Filter by distance threshold
    close_indices = [candidates[i] for i in range(len(candidates)) if distances[i] <= 10]
    
    for idx2 in close_indices:
        # Skip if already assigned
        if df_final.at[idx2, "3d_label"] != -1:
            continue
        
        # Check IOU
        iou = calculate_iou_fast(idx, idx2)
        if iou > 0.3:
            # Recursively stitch this cell forward
            stitch_forward(idx2, current_label)
            # Only match one cell per slice (first match wins)
            break

# Process all cells
for idx in df_final.index:
    # Only process if not yet assigned
    if df_final.at[idx, "3d_label"] == -1:
        unique_label += 1
        stitch_forward(idx, unique_label)

print(f"Total unique 3D cells: {unique_label}")

# create final 3D mask
for idx, row in df_final.iterrows():
    z = int(row["z"])
    mask_3D[z][unique_mask[z] == row["label"]] = row["3d_label"]


Total unique 3D cells: 1232


In [69]:
viewer = napari.Viewer(ndisplay=3)
viewer.add_labels(mask_3D, name="3D_stitched_mask", scale = (2.5,0.64,0.64))
viewer.add_labels(unique_mask, name="original_mask", scale = (2.5,0.64,0.64))

<Labels layer 'original_mask' at 0x1e4c39dc810>